In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import wc26.kinematics.config as config
import wc26.kinematics.nmf_replay as replay

In [ ]:
walking_period_npz_file = Path(
    str(config.KINEMATIC_DATA_PATH_FMT).format(idx=config.WALKING_PERIOD_IDX)
)
joint_angles_dict, thorax_pos_rec, spotlight_trial_dir = replay.load_kinematics_data(
    walking_period_npz_file
)

In [ ]:
sim, fly = replay.set_up_simulation(
    joint_stiffness=5.0,
    joint_damping=0.5,
    actuator_gain=150.0,
    sliding_friction=3.5,
    torsional_friction=0.05,
)

In [ ]:
dof_order_in_sim = [dof.name for dof in fly.get_actuated_jointdofs_order("position")]
sim_timestep = sim.mj_model.opt.timestep
target_angles_arr_sim, rec_match_mask = replay.make_target_joint_angles_array(
    joint_angles_dict,
    dof_order_in_sim,
    rec_fps=config.DATA_FPS,
    sim_timestep=sim_timestep,
)

In [ ]:
sim_results = replay.run_simulation(sim, fly, target_angles_arr_sim)
sim.renderer.show_in_notebook()

In [ ]:
thorax_pos_sim = sim_results["thorax_pos"][rec_match_mask, :2]
thorax_pos_sim -= thorax_pos_sim[0, :]
R, t, thorax_pos_rec_aligned, align_metrics = replay.align_2d_traj(
    thorax_pos_rec, thorax_pos_sim, anchor_origin=True
)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(thorax_pos_rec_aligned[:, 0], thorax_pos_rec_aligned[:, 1], label="Recorded")
ax.plot(thorax_pos_sim[:, 0], thorax_pos_sim[:, 1], label="Simulated")
ax.scatter([0], [0], color="black", zorder=100, label="Origin")
ax.set_aspect("equal")
ax.legend()

In [ ]:
def compute_traj_len(traj):
    return np.linalg.norm(np.diff(traj, axis=0), axis=1).sum()


from scipy.signal import savgol_filter

dt = 1 / config.DATA_FPS
thorax_vel_sim = savgol_filter(
    thorax_pos_sim, window_length=67, polyorder=2, deriv=1, delta=dt, axis=0
)
thorax_vel_rec = savgol_filter(
    thorax_pos_rec_aligned, window_length=67, polyorder=2, deriv=1, delta=dt, axis=0
)

thorax_pos_sim_smooth = np.cumsum(thorax_vel_sim, axis=0) * dt
thorax_pos_rec_smooth = np.cumsum(thorax_vel_rec, axis=0) * dt

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(thorax_pos_rec_aligned[:, 0], thorax_pos_rec_aligned[:, 1], label="Recorded (raw)")
ax.plot(thorax_pos_sim[:, 0], thorax_pos_sim[:, 1], label="Simulated (raw)")
ax.plot(thorax_pos_rec_smooth[:, 0], thorax_pos_rec_smooth[:, 1], label="Recorded (smoothed)")
ax.plot(thorax_pos_sim_smooth[:, 0], thorax_pos_sim_smooth[:, 1], label="Simulated (smoothed)")
ax.scatter([0], [0], color="black", zorder=100, label="Origin")
ax.set_aspect("equal")
ax.legend()

fig, axes = plt.subplots(2, 1, figsize=(6, 6))
axes[0].plot(thorax_vel_rec[:, 0], label="Recorded")
axes[0].plot(thorax_vel_sim[:, 0], label="Simulated")
axes[0].set_title("Thorax Velocity X")
axes[0].legend()
axes[1].plot(thorax_vel_rec[:, 1], label="Recorded")
axes[1].plot(thorax_vel_sim[:, 1], label="Simulated")
axes[1].set_title("Thorax Velocity Y")


ruggedness_sim = compute_traj_len(thorax_pos_sim) / compute_traj_len(thorax_pos_sim_smooth)
ruggedness_rec = compute_traj_len(thorax_pos_rec_aligned) / compute_traj_len(thorax_pos_rec_smooth)
print(f"Ruggedness - Simulated: {ruggedness_sim:.3f}, Recorded: {ruggedness_rec:.3f}")

In [ ]:
from scipy.signal import savgol_filter
from scipy.fft import rfft, rfftfreq

dt = 1 / config.DATA_FPS
thorax_vel_sim = savgol_filter(
    thorax_pos_sim, window_length=33, polyorder=2, deriv=1, delta=dt, axis=0
)
thorax_vel_rec = savgol_filter(
    thorax_pos_rec_aligned, window_length=33, polyorder=2, deriv=1, delta=dt, axis=0
)

fig, ax = plt.subplots(2, 1, figsize=(6, 6))
ax[0].plot(thorax_vel_rec[:, 0], label="Recorded")
ax[0].plot(thorax_vel_sim[:, 0], label="Simulated")
ax[0].set_title("X Velocity")
ax[0].legend()
ax[1].plot(thorax_vel_rec[:, 1], label="Recorded")
ax[1].plot(thorax_vel_sim[:, 1], label="Simulated")
ax[1].set_title("Y Velocity")

n = thorax_vel_rec.shape[0]
freqs = rfftfreq(n, dt)
fft_rec_x = rfft(thorax_vel_rec[:, 0])
fft_rec_y = rfft(thorax_vel_rec[:, 1])
fft_sim_x = rfft(thorax_vel_sim[:, 0])
fft_sim_y = rfft(thorax_vel_sim[:, 1])
magnitude_rec_x = np.abs(fft_rec_x)
magnitude_rec_y = np.abs(fft_rec_y)
magnitude_sim_x = np.abs(fft_sim_x)
magnitude_sim_y = np.abs(fft_sim_y)

fig, ax = plt.subplots(2, 1, figsize=(6, 6))
ax[0].plot(freqs, magnitude_rec_x, label="Recorded")
ax[0].plot(freqs, magnitude_sim_x, label="Simulated")
ax[0].set_title("FFT of X Velocity")
ax[0].set_xlim(0, 20)
ax[0].legend()
ax[1].plot(freqs, magnitude_rec_y, label="Recorded")
ax[1].plot(freqs, magnitude_sim_y, label="Simulated")
ax[1].set_title("FFT of Y Velocity")
ax[1].set_xlim(0, 20)

In [ ]:
import wc26.kinematics.nmf_replay as replay

replay.optimize_sim_params(
    target_angles_arr_sim,
    thorax_pos_rec,
    rec_match_mask,
    n_trials=500,
    n_jobs=16,
)

In [ ]:
thorax_pos_rec_aligned